In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.impute import SimpleImputer

In [ ]:
SEED = 42

# Extraction of Dataset

In [ ]:
df = pd.read_csv('Base.csv')

In [ ]:
df.shape

Train-test split based on month as done by the paper

In [ ]:
df_train = df[df['month'] < 6]
df_test = df[df['month'] >= 6]

In [ ]:
df_train.shape

In [ ]:
df_train.head()

# Transformations for Features

## Checks for Missingness

From the data dictionary, values of "-1" are missing values. Due to data skewness of the columns with it, the missing values are imputed using medians.

In [ ]:
cols_to_impute = [
    'prev_address_months_count',
    'current_address_months_count',
    'session_length_in_minutes',
    'device_distinct_emails_8w']

for col in cols_to_impute:
    df_train[col] = df_train[col].replace(-1, np.nan)

(df_train.isnull().sum() / len(df_train)).sort_values(ascending=False).head(5)

In [ ]:
median_imputer = SimpleImputer(strategy='median')
df_train[cols_to_impute] = median_imputer.fit_transform(df_train[cols_to_impute])

## Checks for Redundancy

### Columns with No variation

In [ ]:
df_train.describe()

In [ ]:
df_train.describe(exclude='number')

device_fraud_count does not show any variation. The column can be dropped.

### Correlated Features

We determine highly correlated columns for further analysis and potential feature extraction. In this case, pairs with an absolute correlation greater than 0.7 are investigated.

In [ ]:
df_train_enc = pd.get_dummies(df_train, drop_first=True)
df_train_enc.shape

In [ ]:
corr = df_train_enc.corr()

row_indices, col_indices = np.where(((corr < -0.7) | (corr > 0.7)) & (corr < 1))

coordinates = list(zip(row_indices, col_indices))

coordinates

In [ ]:
corr.iloc[11,26]

In [ ]:
corr.index[11]

In [ ]:
corr.index[26]

The only highly correlated pair of columns is velocity_4w (average number of applications given a time period) and month. It would not make sense to extract features as 'month' is just an interval column from 0-7 with no physical interpretation.

## Train-test Split

In [ ]:
X_train = df_train_enc.drop(columns=['fraud_bool','device_fraud_count'])
y_train = df_train_enc['fraud_bool']

df_test_enc = pd.get_dummies(df_test, drop_first=True)
X_test = df_test_enc.drop(columns=['fraud_bool','device_fraud_count'])
y_test = df_test_enc['fraud_bool']

## Balancing of Fraud & Non-Fraud cases


In [ ]:
df_train_enc.fraud_bool.value_counts()

For computational efficiency, Randomized Under Sampling is used to balance the cases

In [ ]:
rus = RandomUnderSampler(random_state=SEED)
X_train, y_train = rus.fit_resample(X_train, y_train)

## Scaling

In [ ]:
scaler = StandardScaler()

In [ ]:
continuous_cols = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

# Exporting of Datasets

In [ ]:
pd.DataFrame(X_train).to_csv('X_train.csv', index=False)
pd.DataFrame(y_train).to_csv('y_train.csv', index=False)
pd.DataFrame(X_test).to_csv('X_test.csv', index=False)
pd.DataFrame(y_test).to_csv('y_test.csv', index=False)